# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Ranking/scoring. Section 6's task-type table maps "which ones first?" to ranking/scoring with a priority score as target and precision@K as metric — that's exactly the Lane 2 question ("which pages should be reviewed first"). It's not classification in the strict sense, because the deliverable an editor needs is an ordered queue, not just a yes/no per page — though the underlying label used to build that score is binary (see section 2), so the model itself is a classifier whose output probability gets used as a ranking score. Worth naming both: classification model, ranking use

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The target is is_declining_label, defined in the pipeline as trend_direction == "down" — true for 16,262 of 30,000 rows (54.2%). This is a defined-rule label, not a purely observed outcome: it's a threshold rule (trend_pct < -20% → "down") applied to a measured quantity (trend_pct, itself computed from impressions_last_30d vs impressions_prev_30d). That distinction matters — the model will learn to reproduce this specific threshold rule's output, not "decline" in some looser real-world sense. Because the label is built from trend_direction/trend_pct, both columns are permanently excluded as features (data-dictionary rule #2) — the model has to predict decline from other signals, not from the arithmetic that defines decline.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

Precision@K, with K set to a realistic weekly editor review capacity (the starter pipeline uses K=50). Precision@K is defensible here because Lane 2's real cost function is asymmetric review time — an editor only ever looks at the top of the queue, so what matters is "of the pages I actually have time to review, how many were really worth it," not accuracy across all 30,000 rows (where the label's 54.2% base rate would make a naive "always predict decline" baseline look artificially decent). Recall on the top-K is worth tracking alongside it, given the earlier finding that false negatives (missed decliners) cost more than false positives.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

 One row = one content page (content_id), belonging to one of 32 clients, measured over a trailing 90-day window ending at export. content_id is unique per row (30,000 unique IDs across 30,000 rows) — no duplicate pages, so no aggregation is needed at this stage, only leakage-safe splitting: since client_id has only 32 distinct values with rows nested inside each, a train/test split must group by client_id, not split individual pages, or the model will leak client-specific baseline behavior across the split.








## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

It might not, and that's the point of building the baseline first (Lane 2's own instruction). A fixed rule like "declining + stale (91+ days)" already isolates a defensible 19.7% of visible pages with zero modeling — that's the bar to beat, not a strawman. ML earns its place only if a model combining several tangled signals (position, CTR, engagement, freshness, content type, search volume) at once outperforms that single-condition rule at precision@K, because those signals don't reduce cleanly to one threshold — a page can be declining and fresh, or stale and stable, and the rule can't weigh them against each other the way a trained model can. The reference pipeline's own precision@50 jump (~0.24 → ~0.74) is the existence proof that this gap is real in this dataset — but that number came from a different pipeline run, not this project's own validated split, so it's a target to test for, not a result to assume.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.